# Experiment Scenario: I5\nThis experiment executes the pipeline on the BISINDO dataset (Signer Dependent) following the instructions in PANDUAN_COLAB.md.\n**Note:** Please ensure your Google Colab runtime is set to GPU (Runtime > Change runtime type > Hardware accelerator: T4/V100/A100 GPU) before running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Initial Setup
Connect your Google Drive to securely save your progress, clone the repository, and navigate to the project directory.

In [ ]:
# Clone the repository (replace the URL if you are using your own fork)
!git clone https://github.com/MahardikaPratama/MSLR_ICCV2025.git

# Navigate to the project directory
%cd MSLR_ICCV2025

# Install required Python dependencies
!pip install -r requirements.txt

## 2. Install Dependencies
Install `ctcdecode` and `sclite` for CTC decoding and alignment evaluations.

In [ ]:
!git clone --recursive https://github.com/WayenVan/ctcdecode.git
%cd ctcdecode
!pip install .
%cd /content/MSLR_ICCV2025

In [ ]:
!mkdir -p ./software
!git clone https://github.com/usnistgov/SCTK.git
%cd SCTK
!make config
!make all
!make check
!make install
!make doc
%cd ..
!ln -s $(pwd)/SCTK/bin/sclite ./software/sclite

## 3. Dataset Preparation
Download the BISINDO dataset using `gdown` and run the preprocessing script.

In [ ]:
!pip install gdown
!mkdir -p ./datasets/mslr2025

# Download the complete dataset from the Google Drive Folder
!gdown --folder "1m9kV0-32o8ET5pLAlfeTUewr9zgOM-j-" -O ./datasets_temp
!mv ./datasets_temp/* ./datasets/mslr2025/
!rm -rf ./datasets_temp

In [ ]:
%cd preprocess/mslr2025
!python mslr_process.py
%cd ../../

## 4. Training Process
Ensure you have configured the data augmentation strategy in `datasets/skeleton_feeder.py` (around line 194) before proceeding.
Run the cell below to start training for the Signer Dependent task:

In [ ]:
!python main.py --config ./configs/experiment_configs/input_signal_selection/I5.yaml

## 5. Testing Process
Replace the `--load-weights` path with the correct model weight file (`.pth` or `.pt`) you want to evaluate.

In [ ]:
!python main.py \\\n    --config ./configs/experiment_configs/input_signal_selection/I5.yaml \\\n    --phase test \\\n    --load-weights ./work_dir/I5/[REPLACE_WITH_BEST_MODEL_NAME].pt

## 6. Download Experiment Results
Run the cell below to archive the `work_dir` folder. The outputs are separated into two distinct `.zip` files: one containing only the model weights (`experiment_models.zip`) and another containing the experiment logs and results (`experiment_results.zip`).

In [ ]:
import os
from google.colab import files

print("Zipping experiment results (excluding model weights)...")
!zip -r experiment_results.zip ./work_dir/ -x "*.pt" "*.pth"

print("Downloading files to your local machine...")
if os.path.exists('experiment_results.zip'):
    files.download('experiment_results.zip')

In [ ]:
import os
from google.colab import files

print("Zipping model weights...")
!zip -r experiment_models.zip ./work_dir/ -i "*.pt" "*.pth"

if os.path.exists('experiment_models.zip'):
    files.download('experiment_models.zip')